# Pseudo-Top Interaction XGBoost

This notebook tests whether pseudo-top uncertainty helps when it is expressed as targeted interaction features inside the primary XGBoost mean model.

## 1. Purpose And Leakage Rules

The previous pseudo-top notebooks found meaningful fold-safe top inference, but raw pseudo-top columns and post-hoc residual gates did not beat the strict local reference. This pass keeps the same leakage discipline and asks a narrower question: can XGBoost use pseudo-top uncertainty as an interaction/regime signal in the mean function?

Direct formation columns are used only as local pseudo-top labels and oracle-free diagnostics. Legal candidates receive fold-predicted pseudo-top values plus deterministic interactions with progress, distance, analog stress, and typewell fit.

## 2. Setup

In [1]:
"Configure imports, paths, and deterministic settings."

from pathlib import Path
import json
import shutil
import time

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)

ROOT = Path.cwd()
if not (ROOT / "context").exists():
    ROOT = next(parent for parent in ROOT.parents if (parent / "context").exists())

RUN_DATE = "2026-06-30"
RUN_TAG = "pseudo_top_interaction_xgboost"
RUN_ID = f"{RUN_DATE}__{RUN_TAG}"
SEED = 20260630
FORMATION_COLS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
PRIMARY_FOLD = "strict_hybrid_fold"
STRESS_FOLD = "analog_distance_fold"
MAX_TARGET_ROWS_PER_WELL = 120
ENSEMBLE_SEEDS = [20260630, 20260631]

DATA_DIR = ROOT / "data" / "raw"
TRAIN_DIR = DATA_DIR / "train"
TABLE_DIR = ROOT / "artifacts" / "tables"
REPORT_DIR = ROOT / "artifacts" / "reports"
RUN_DIR = ROOT / "artifacts" / "logs" / "runs" / RUN_ID
for directory in [TABLE_DIR, REPORT_DIR, RUN_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "run_id": RUN_ID,
    "seed": SEED,
    "formation_cols": FORMATION_COLS,
    "primary_fold": PRIMARY_FOLD,
    "stress_fold": STRESS_FOLD,
    "max_target_rows_per_well": MAX_TARGET_ROWS_PER_WELL,
    "ensemble_seeds": ENSEMBLE_SEEDS,
    "current_public_best_rmse": 14.472,
    "best_xgb_known_state_local_rmse": 13.4334,
    "best_xgb_known_state_stress_rmse": 13.7533,
    "promotion_margin": 0.05,
}
with (RUN_DIR / "config.json").open("w", encoding="utf-8") as handle:
    json.dump(CONFIG, handle, indent=2)

print(ROOT)
print(RUN_DIR)

C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\logs\runs\2026-06-30__pseudo_top_interaction_xgboost


## 3. Data Loading And Label Audit

In [2]:
"Build sampled target-zone rows with legal predictors and direct top-elevation labels."

folds = pd.read_csv(TABLE_DIR / "validation__strict_cv_fold_assignments__2026-06-26.csv")
descriptors = pd.read_csv(TABLE_DIR / "model__typewell_alignment_feature_descriptors__2026-06-23.csv")
well_meta = folds.merge(descriptors, on="wellname", how="left")
well_meta = well_meta.drop_duplicates("wellname").reset_index(drop=True)

DESCRIPTOR_COLS = [
    "known_rows", "known_tvt_span", "known_md_span", "known_gr_mean_last300", "known_gr_std_last300", "known_gr_slope_last300",
    "xy_span_known", "z_drop_known", "typewell_corr_last300", "typewell_mae_last300", "typewell_best_lag_corr",
    "typewell_best_lag_tvt", "typewell_best_lag_mae", "selected_analog_distance", "extrapolation_stress_score",
    "analog_stress_score", "combined_stress_score", "descriptor_cluster",
]


def safe_divide(num, den):
    return num / (den + 1e-9)


def load_typewell(directory, wellname):
    path = Path(directory) / f"{wellname}__typewell.csv"
    if not path.exists():
        return pd.DataFrame(columns=["TVT", "GR", "Geology"])
    return pd.read_csv(path)


def interp_typewell_gr(typewell, tvt_values):
    if typewell.empty or not {"TVT", "GR"}.issubset(typewell.columns):
        return np.full(len(tvt_values), np.nan)
    clean = typewell[["TVT", "GR"]].dropna().sort_values("TVT")
    if len(clean) < 5:
        return np.full(len(tvt_values), np.nan)
    return np.interp(np.asarray(tvt_values, dtype=float), clean["TVT"].to_numpy(), clean["GR"].to_numpy(), left=np.nan, right=np.nan)


def build_target_rows(csv_path, meta_row):
    df = pd.read_csv(csv_path).sort_values("MD").reset_index(drop=True)
    known_mask = df["TVT_input"].notna()
    if known_mask.sum() == 0 or (~known_mask).sum() == 0:
        return pd.DataFrame()
    last_known_idx = int(np.flatnonzero(known_mask.to_numpy())[-1])
    known = df.loc[:last_known_idx].copy()
    target = df.loc[last_known_idx + 1:].copy()
    target = target.loc[target["TVT"].notna()].copy()
    if target.empty:
        return pd.DataFrame()
    if len(target) > MAX_TARGET_ROWS_PER_WELL:
        sample_idx = np.linspace(0, len(target) - 1, MAX_TARGET_ROWS_PER_WELL).round().astype(int)
        target = target.iloc[np.unique(sample_idx)].copy()

    typewell = load_typewell(csv_path.parent, meta_row["wellname"])
    last = known.iloc[-1]
    first = df.iloc[0]
    known_gr = known["GR"].astype(float).clip(0, 300)
    known_gr_finite = known_gr.dropna()
    gr_fill = float(known_gr.tail(300).median()) if known_gr.notna().any() else 0.0
    target_gr = target["GR"].astype(float).clip(0, 300).fillna(gr_fill)

    known_md_span_state = float(known["MD"].iloc[-1] - known["MD"].iloc[0]) if len(known) > 1 else 0.0
    known_tvt_delta_state = float(known["TVT_input"].iloc[-1] - known["TVT_input"].iloc[0]) if len(known) > 1 else 0.0
    known_tvt_slope_state = known_tvt_delta_state / max(abs(known_md_span_state), 1.0)
    known_xy_span_state = float(np.hypot(known["X"].iloc[-1] - known["X"].iloc[0], known["Y"].iloc[-1] - known["Y"].iloc[0])) if len(known) > 1 else 0.0
    known_z_drop_state = float(known["Z"].iloc[-1] - known["Z"].iloc[0]) if len(known) > 1 else 0.0
    known_gr_coverage_state = float(known_gr.notna().mean()) if len(known_gr) else 0.0
    known_gr_last_state = float(known_gr_finite.iloc[-1]) if len(known_gr_finite) else gr_fill
    known_gr_iqr_state = float(known_gr_finite.quantile(0.75) - known_gr_finite.quantile(0.25)) if len(known_gr_finite) >= 4 else 0.0

    target = target.reset_index(drop=False).rename(columns={"index": "row_idx_original"})
    out = pd.DataFrame({
        "wellname": meta_row["wellname"],
        "row_idx_original": target["row_idx_original"].astype(float),
        "MD": target["MD"].astype(float),
        "X": target["X"].astype(float),
        "Y": target["Y"].astype(float),
        "Z": target["Z"].astype(float),
        "GR": target_gr.to_numpy(),
        "TVT": target["TVT"].astype(float),
        "last_known_tvt_feature": float(last["TVT"]),
        "last_known_md": float(last["MD"]),
        "last_known_x": float(last["X"]),
        "last_known_y": float(last["Y"]),
        "last_known_z": float(last["Z"]),
        "first_md": float(first["MD"]),
        "state_known_tvt_last": float(last["TVT_input"] if pd.notna(last.get("TVT_input", np.nan)) else last["TVT"]),
        "state_known_tvt_slope": known_tvt_slope_state,
        "state_known_gr_coverage": known_gr_coverage_state,
        "state_known_gr_last": known_gr_last_state,
        "state_known_gr_iqr": known_gr_iqr_state,
        "state_known_xy_span": known_xy_span_state,
        "state_known_z_drop": known_z_drop_state,
        "state_known_md_span": known_md_span_state,
    })
    out["target_delta"] = out["TVT"] - out["last_known_tvt_feature"]
    out["md_from_last_known"] = out["MD"] - out["last_known_md"]
    out["md_from_start"] = out["MD"] - out["first_md"]
    out["eval_progress"] = safe_divide(np.arange(1, len(out) + 1), len(out))
    out["eval_progress_sq"] = out["eval_progress"] ** 2
    out["dx_from_last"] = out["X"] - out["last_known_x"]
    out["dy_from_last"] = out["Y"] - out["last_known_y"]
    out["dz_from_last"] = out["Z"] - out["last_known_z"]
    out["horizontal_from_last"] = np.sqrt(out["dx_from_last"] ** 2 + out["dy_from_last"] ** 2)
    out["distance_from_last"] = np.sqrt(out["horizontal_from_last"] ** 2 + out["dz_from_last"] ** 2)
    out["distance_log1p"] = np.log1p(out["distance_from_last"].clip(lower=0))
    out["tvt_linear_proxy"] = out["last_known_tvt_feature"] + out["md_from_last_known"]
    best_lag = float(meta_row.get("typewell_best_lag_tvt", 0.0) if pd.notna(meta_row.get("typewell_best_lag_tvt", np.nan)) else 0.0)
    out["typewell_gr_linear_tvt"] = interp_typewell_gr(typewell, out["tvt_linear_proxy"].to_numpy())
    out["typewell_gr_lag_best"] = interp_typewell_gr(typewell, out["tvt_linear_proxy"].to_numpy() + best_lag)
    out["gr_minus_typewell_linear"] = out["GR"] - out["typewell_gr_linear_tvt"]
    out["gr_minus_typewell_bestlag"] = out["GR"] - out["typewell_gr_lag_best"]

    for formation in FORMATION_COLS:
        out[f"formation_top_{formation}"] = target[formation].astype(float).to_numpy()
        out[f"label_signed_offset_{formation}"] = out["Z"] - out[f"formation_top_{formation}"]
        out[f"label_abs_offset_{formation}"] = np.abs(out[f"label_signed_offset_{formation}"])

    abs_label_cols = [f"label_abs_offset_{formation}" for formation in FORMATION_COLS]
    out["label_nearest_abs_offset"] = out[abs_label_cols].min(axis=1)
    out["label_nearest_formation"] = out[abs_label_cols].idxmin(axis=1).str.replace("label_abs_offset_", "", regex=False)
    out["label_mean_abs_offset"] = out[abs_label_cols].mean(axis=1)

    for col in [PRIMARY_FOLD, STRESS_FOLD, "descriptor_cluster_fold", "extrapolation_fold"]:
        out[col] = int(meta_row[col])
    for col in DESCRIPTOR_COLS:
        out[col] = float(meta_row[col]) if pd.notna(meta_row[col]) else np.nan
    return out


frames = []
for meta_row in well_meta.to_dict(orient="records"):
    csv_path = TRAIN_DIR / f"{meta_row['wellname']}__horizontal_well.csv"
    if csv_path.exists():
        frame = build_target_rows(csv_path, meta_row)
        if not frame.empty:
            frames.append(frame)

data = pd.concat(frames, ignore_index=True)

SAFE_ROW_COLS = [
    "row_idx_original", "MD", "X", "Y", "Z", "GR", "last_known_tvt_feature", "last_known_md",
    "md_from_last_known", "md_from_start", "eval_progress", "eval_progress_sq",
    "dx_from_last", "dy_from_last", "dz_from_last", "horizontal_from_last", "distance_from_last", "distance_log1p",
]
SAFE_FEATURE_COLS = [col for col in [*SAFE_ROW_COLS, *DESCRIPTOR_COLS] if col in data.columns]
TYPEWELL_FEATURE_COLS = [
    "tvt_linear_proxy", "typewell_gr_linear_tvt", "typewell_gr_lag_best", "gr_minus_typewell_linear", "gr_minus_typewell_bestlag",
]
TYPEWELL_FEATURE_COLS = [col for col in TYPEWELL_FEATURE_COLS if col in data.columns]
STATE_FEATURE_COLS = [
    "state_known_tvt_last", "state_known_tvt_slope", "state_known_gr_coverage", "state_known_gr_last",
    "state_known_gr_iqr", "state_known_xy_span", "state_known_z_drop", "state_known_md_span",
]
DIRECT_FORMATION_FEATURE_COLS = [
    *[f"label_signed_offset_{formation}" for formation in FORMATION_COLS],
    *[f"label_abs_offset_{formation}" for formation in FORMATION_COLS],
    "label_nearest_abs_offset", "label_mean_abs_offset",
]

top_label_summary = []
for formation in FORMATION_COLS:
    values = data[f"formation_top_{formation}"].astype(float)
    offsets = data[f"label_signed_offset_{formation}"].astype(float)
    top_label_summary.append({
        "formation": formation,
        "top_mean": float(values.mean()),
        "top_std": float(values.std()),
        "top_min": float(values.min()),
        "top_max": float(values.max()),
        "signed_offset_mean": float(offsets.mean()),
        "signed_offset_std": float(offsets.std()),
        "abs_offset_mean": float(np.abs(offsets).mean()),
    })
top_label_summary = pd.DataFrame(top_label_summary)
label_counts = data["label_nearest_formation"].value_counts().rename_axis("formation").reset_index(name="nearest_label_rows")
top_label_summary = top_label_summary.merge(label_counts, on="formation", how="left").fillna({"nearest_label_rows": 0})

label_path = TABLE_DIR / "model__pseudo_top_interaction_xgboost_top_label_summary__2026-06-30.csv"
top_label_summary.to_csv(label_path, index=False)
shutil.copy2(label_path, RUN_DIR / label_path.name)

feature_preview_path = TABLE_DIR / "model__pseudo_top_interaction_xgboost_feature_preview__2026-06-30.csv"
data.head(500).to_csv(feature_preview_path, index=False)
shutil.copy2(feature_preview_path, RUN_DIR / feature_preview_path.name)

print(data.shape)
print("safe_features", len(SAFE_FEATURE_COLS))
print(top_label_summary.to_string(index=False))

(92760, 81)
safe_features 36
formation     top_mean    top_std   top_min  top_max  signed_offset_mean  signed_offset_std  abs_offset_mean  nearest_label_rows
     ANCC -9008.099049 633.914859 -10647.31 -7301.76         -416.172460          55.411650       416.172460                 0.0
    ASTNU -9190.780447 639.418693 -10852.79 -7509.20         -244.653750          63.560087       244.653750                 0.0
    ASTNL -9255.299324 624.857096 -10865.58 -7614.43         -180.134874          51.475069       180.135674              2455.0
    EGFDU -9349.765034 619.827567 -10887.58 -7704.03          -85.669164          43.889718        91.053770              3919.0
    EGFDL -9391.937722 619.776843 -10932.35 -7731.61          -43.798726          44.572447        55.375403             50858.0
     BUDA -9516.381964 633.862561 -11104.73 -7835.57           80.947766          45.807212        80.953400             35528.0


## 4. Fold-Safe Pseudo-Top Feature Builders

In [3]:
"Train fold-safe top-elevation ensembles and derive uncertainty, margin, and nearest-top features."

PSEUDO_TOP_PREDICTOR_COLS = [
    *SAFE_FEATURE_COLS,
    *TYPEWELL_FEATURE_COLS,
    *STATE_FEATURE_COLS,
]
PSEUDO_TOP_PREDICTOR_COLS = list(dict.fromkeys([col for col in PSEUDO_TOP_PREDICTOR_COLS if col in data.columns]))

pseudo_params = {
    "objective": "reg:squarederror",
    "n_estimators": 220,
    "learning_rate": 0.040,
    "max_depth": 3,
    "min_child_weight": 90,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.20,
    "reg_lambda": 10.0,
    "tree_method": "hist",
    "n_jobs": 4,
    "verbosity": 0,
}


def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


def softmax_neg_abs(abs_matrix, scale=50.0):
    logits = -abs_matrix / scale
    logits = logits - np.nanmax(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / np.clip(exp_logits.sum(axis=1, keepdims=True), 1e-12, None)


def add_pseudo_features_for_fold_column(frame, fold_col):
    out = frame.copy()
    proxy_quality_rows = []
    for formation in FORMATION_COLS:
        out[f"pseudo_top_elev_{formation}__{fold_col}"] = np.nan
        out[f"pseudo_top_std_{formation}__{fold_col}"] = np.nan

    X_all = out[PSEUDO_TOP_PREDICTOR_COLS].replace([np.inf, -np.inf], np.nan)
    for fold in sorted(out[fold_col].dropna().unique()):
        valid_mask = out[fold_col].eq(fold)
        train_mask = ~valid_mask
        X_train = X_all.loc[train_mask]
        X_valid = X_all.loc[valid_mask]
        for formation in FORMATION_COLS:
            target_col = f"formation_top_{formation}"
            label_train_mask = train_mask & out[target_col].notna()
            label_valid_mask = valid_mask & out[target_col].notna()
            seed_predictions = []
            if label_train_mask.sum() >= 20:
                X_label_train = X_all.loc[label_train_mask]
                y_train = out.loc[label_train_mask, target_col].astype(float)
                for seed in ENSEMBLE_SEEDS:
                    model = XGBRegressor(**pseudo_params, random_state=seed)
                    model.fit(X_label_train, y_train, verbose=False)
                    seed_predictions.append(model.predict(X_valid))
                pred_matrix = np.column_stack(seed_predictions)
                pred_mean = pred_matrix.mean(axis=1)
                pred_std = pred_matrix.std(axis=1)
            else:
                pred_mean = np.full(valid_mask.sum(), float(out[target_col].median()))
                pred_std = np.zeros(valid_mask.sum())
            out.loc[valid_mask, f"pseudo_top_elev_{formation}__{fold_col}"] = pred_mean
            out.loc[valid_mask, f"pseudo_top_std_{formation}__{fold_col}"] = pred_std
            if label_valid_mask.sum() > 0:
                valid_index = out.loc[valid_mask].index
                pred_series = pd.Series(pred_mean, index=valid_index).loc[label_valid_mask]
                y_valid = out.loc[label_valid_mask, target_col].astype(float)
                true_offset = out.loc[label_valid_mask, "Z"].to_numpy(float) - y_valid.to_numpy(float)
                pred_offset = out.loc[label_valid_mask, "Z"].to_numpy(float) - pred_series.to_numpy(float)
                corr = float(np.corrcoef(y_valid.to_numpy(float), pred_series.to_numpy(float))[0, 1]) if len(y_valid) > 2 else np.nan
                top_rmse = rmse(y_valid, pred_series)
                top_mae = float(mean_absolute_error(y_valid, pred_series))
                signed_offset_rmse = rmse(true_offset, pred_offset)
            else:
                corr = np.nan
                top_rmse = np.nan
                top_mae = np.nan
                signed_offset_rmse = np.nan
            proxy_quality_rows.append({
                "fold_column": fold_col,
                "fold": int(fold),
                "formation": formation,
                "n_train": int(label_train_mask.sum()),
                "n_valid": int(label_valid_mask.sum()),
                "top_rmse": top_rmse,
                "top_mae": top_mae,
                "top_corr": corr,
                "signed_offset_rmse": signed_offset_rmse,
                "mean_ensemble_std": float(np.mean(pred_std)),
            })

    for formation in FORMATION_COLS:
        top_col = f"pseudo_top_elev_{formation}__{fold_col}"
        std_col = f"pseudo_top_std_{formation}__{fold_col}"
        out[top_col] = out[top_col].fillna(float(out[f"formation_top_{formation}"].median()))
        out[std_col] = out[std_col].fillna(float(out[std_col].median()) if out[std_col].notna().any() else 0.0)
        out[f"pseudo_signed_offset_{formation}__{fold_col}"] = out["Z"] - out[top_col]
        out[f"pseudo_abs_offset_{formation}__{fold_col}"] = np.abs(out[f"pseudo_signed_offset_{formation}__{fold_col}"])

    abs_cols = [f"pseudo_abs_offset_{formation}__{fold_col}" for formation in FORMATION_COLS]
    std_cols = [f"pseudo_top_std_{formation}__{fold_col}" for formation in FORMATION_COLS]
    abs_matrix = out[abs_cols].to_numpy(float)
    probs = softmax_neg_abs(abs_matrix)
    sorted_abs = np.sort(abs_matrix, axis=1)
    sorted_probs = np.sort(probs, axis=1)[:, ::-1]
    nearest_idx = np.argmin(abs_matrix, axis=1)

    out[f"pseudo_nearest_abs_offset__{fold_col}"] = sorted_abs[:, 0]
    out[f"pseudo_second_abs_offset__{fold_col}"] = sorted_abs[:, 1]
    out[f"pseudo_top_margin__{fold_col}"] = out[f"pseudo_second_abs_offset__{fold_col}"] - out[f"pseudo_nearest_abs_offset__{fold_col}"]
    out[f"pseudo_top_confidence__{fold_col}"] = 1.0 / (1.0 + np.exp(-out[f"pseudo_top_margin__{fold_col}"].clip(-100, 100) / 25.0))
    out[f"pseudo_near_boundary_25__{fold_col}"] = (out[f"pseudo_nearest_abs_offset__{fold_col}"] <= 25.0).astype(float)
    out[f"pseudo_uncertain_boundary__{fold_col}"] = ((out[f"pseudo_nearest_abs_offset__{fold_col}"] <= 50.0) & (out[f"pseudo_top_margin__{fold_col}"] <= 25.0)).astype(float)
    out[f"pseudo_mean_abs_offset__{fold_col}"] = abs_matrix.mean(axis=1)
    out[f"pseudo_min_std__{fold_col}"] = out[std_cols].min(axis=1)
    out[f"pseudo_mean_std__{fold_col}"] = out[std_cols].mean(axis=1)
    out[f"pseudo_top1_prob__{fold_col}"] = sorted_probs[:, 0]
    out[f"pseudo_top2_prob__{fold_col}"] = sorted_probs[:, 1]
    out[f"pseudo_top_entropy__{fold_col}"] = -(probs * np.log(probs + 1e-12)).sum(axis=1)
    out[f"pseudo_nearest_formation__{fold_col}"] = [FORMATION_COLS[int(idx)] for idx in nearest_idx]
    out[f"pseudo_nearest_matches_label__{fold_col}"] = (out[f"pseudo_nearest_formation__{fold_col}"] == out["label_nearest_formation"]).astype(float)

    return out, pd.DataFrame(proxy_quality_rows)


pseudo_feature_cols_by_fold = {}
pseudo_uncertainty_cols_by_fold = {}
quality_parts = []
for fold_col in [PRIMARY_FOLD, STRESS_FOLD]:
    data, quality = add_pseudo_features_for_fold_column(data, fold_col)
    quality_parts.append(quality)
    pseudo_feature_cols_by_fold[fold_col] = [
        *[f"pseudo_signed_offset_{formation}__{fold_col}" for formation in FORMATION_COLS],
        *[f"pseudo_abs_offset_{formation}__{fold_col}" for formation in FORMATION_COLS],
        f"pseudo_nearest_abs_offset__{fold_col}",
        f"pseudo_second_abs_offset__{fold_col}",
        f"pseudo_mean_abs_offset__{fold_col}",
    ]
    pseudo_uncertainty_cols_by_fold[fold_col] = [
        *[f"pseudo_top_std_{formation}__{fold_col}" for formation in FORMATION_COLS],
        f"pseudo_top_margin__{fold_col}",
        f"pseudo_top_confidence__{fold_col}",
        f"pseudo_near_boundary_25__{fold_col}",
        f"pseudo_uncertain_boundary__{fold_col}",
        f"pseudo_min_std__{fold_col}",
        f"pseudo_mean_std__{fold_col}",
        f"pseudo_top1_prob__{fold_col}",
        f"pseudo_top2_prob__{fold_col}",
        f"pseudo_top_entropy__{fold_col}",
    ]

proxy_quality = pd.concat(quality_parts, ignore_index=True)
nearest_quality = []
for fold_col in [PRIMARY_FOLD, STRESS_FOLD]:
    nearest_quality.append({
        "fold_column": fold_col,
        "fold": "all",
        "formation": "nearest_classifier",
        "n_train": np.nan,
        "n_valid": int(len(data)),
        "top_rmse": np.nan,
        "top_mae": np.nan,
        "top_corr": np.nan,
        "signed_offset_rmse": np.nan,
        "mean_ensemble_std": float(data[f"pseudo_mean_std__{fold_col}"].mean()),
        "nearest_accuracy": float(data[f"pseudo_nearest_matches_label__{fold_col}"].mean()),
    })
proxy_quality["nearest_accuracy"] = np.nan
proxy_quality = pd.concat([proxy_quality, pd.DataFrame(nearest_quality)], ignore_index=True)

proxy_quality_path = TABLE_DIR / "model__pseudo_top_interaction_xgboost_proxy_quality__2026-06-30.csv"
proxy_quality.to_csv(proxy_quality_path, index=False)
shutil.copy2(proxy_quality_path, RUN_DIR / proxy_quality_path.name)
proxy_quality.groupby(["fold_column", "formation"], dropna=False).agg(top_rmse=("top_rmse", "mean"), top_mae=("top_mae", "mean"), mean_ensemble_std=("mean_ensemble_std", "mean"), nearest_accuracy=("nearest_accuracy", "mean")).reset_index()

,fold_column,formation,top_rmse,top_mae,mean_ensemble_std,nearest_accuracy
0,analog_distance_fold,ANCC,36.448121,27.561076,2.225590,NaN
1,analog_distance_fold,ASTNL,36.512561,27.158327,2.604606,NaN
2,analog_distance_fold,ASTNU,38.155576,28.416605,2.408923,NaN
3,analog_distance_fold,BUDA,34.737212,25.140603,2.262772,NaN
4,analog_distance_fold,EGFDL,34.746888,25.533790,2.353502,NaN
5,analog_distance_fold,EGFDU,34.494617,25.287956,2.264115,NaN
6,analog_distance_fold,nearest_classifier,NaN,NaN,2.353398,0.579743
7,strict_hybrid_fold,ANCC,35.638111,26.984800,1.995119,NaN
8,strict_hybrid_fold,ASTNL,35.637506,26.891537,2.281886,NaN
9,strict_hybrid_fold,ASTNU,37.183606,27.877467,2.611257,NaN


## 5. Interaction Feature Families

In [4]:
"Create controlled pseudo-top interaction families for strict and stress folds."


def add_interaction_features(frame, fold_col):
    out = frame
    margin = out[f"pseudo_top_margin__{fold_col}"]
    entropy = out[f"pseudo_top_entropy__{fold_col}"]
    confidence = out[f"pseudo_top_confidence__{fold_col}"]
    nearest = out[f"pseudo_nearest_abs_offset__{fold_col}"]
    mean_std = out[f"pseudo_mean_std__{fold_col}"]
    analog = out["selected_analog_distance"].fillna(out["selected_analog_distance"].median())
    typewell_mae = out["typewell_mae_last300"].fillna(out["typewell_mae_last300"].median())
    distance = out["distance_log1p"]
    progress = out["eval_progress"]
    stress = out["combined_stress_score"].fillna(out["combined_stress_score"].median())

    out[f"ix_margin_x_progress__{fold_col}"] = margin * progress
    out[f"ix_entropy_x_analog__{fold_col}"] = entropy * analog
    out[f"ix_nearest_x_distance__{fold_col}"] = nearest * distance
    out[f"ix_confidence_x_typewell_mae__{fold_col}"] = confidence * typewell_mae
    out[f"ix_std_x_stress__{fold_col}"] = mean_std * stress
    out[f"ix_confidence_late__{fold_col}"] = confidence * (progress >= 0.66).astype(float)
    out[f"ix_uncertain_early__{fold_col}"] = (1.0 - confidence) * (progress <= 0.33).astype(float)
    out[f"ix_boundary_mid__{fold_col}"] = (nearest <= 35.0).astype(float) * ((progress > 0.33) & (progress < 0.66)).astype(float)
    out[f"ix_margin_x_distance__{fold_col}"] = margin * distance
    out[f"ix_entropy_x_progress__{fold_col}"] = entropy * progress
    return out


interaction_cols_by_fold = {}
for active_fold in [PRIMARY_FOLD, STRESS_FOLD]:
    data = add_interaction_features(data, active_fold)
    interaction_cols_by_fold[active_fold] = [
        f"ix_margin_x_progress__{active_fold}",
        f"ix_entropy_x_analog__{active_fold}",
        f"ix_nearest_x_distance__{active_fold}",
        f"ix_confidence_x_typewell_mae__{active_fold}",
        f"ix_std_x_stress__{active_fold}",
        f"ix_confidence_late__{active_fold}",
        f"ix_uncertain_early__{active_fold}",
        f"ix_boundary_mid__{active_fold}",
        f"ix_margin_x_distance__{active_fold}",
        f"ix_entropy_x_progress__{active_fold}",
    ]

print(interaction_cols_by_fold[PRIMARY_FOLD])
data[interaction_cols_by_fold[PRIMARY_FOLD]].describe().T

['ix_margin_x_progress__strict_hybrid_fold', 'ix_entropy_x_analog__strict_hybrid_fold', 'ix_nearest_x_distance__strict_hybrid_fold', 'ix_confidence_x_typewell_mae__strict_hybrid_fold', 'ix_std_x_stress__strict_hybrid_fold', 'ix_confidence_late__strict_hybrid_fold', 'ix_uncertain_early__strict_hybrid_fold', 'ix_boundary_mid__strict_hybrid_fold', 'ix_margin_x_distance__strict_hybrid_fold', 'ix_entropy_x_progress__strict_hybrid_fold']


,count,mean,std,min,25%,50%,75%,max
ix_margin_x_progress__strict_hybrid_fold,92760.0,14.402268,14.165752,0.000015,3.790555,10.270838,21.388525,176.656120
ix_entropy_x_analog__strict_hybrid_fold,92760.0,0.485001,0.206682,0.075705,0.354394,0.449435,0.569785,2.806255
ix_nearest_x_distance__strict_hybrid_fold,92760.0,314.036583,141.195713,0.006790,221.302287,329.599586,418.613767,942.978625
ix_confidence_x_typewell_mae__strict_hybrid_fold,92760.0,6.268853,3.007730,1.223802,4.239759,5.633443,7.505973,26.606046
ix_std_x_stress__strict_hybrid_fold,92760.0,1.182516,0.816258,0.053494,0.675472,0.996165,1.437042,11.453132
ix_confidence_late__strict_hybrid_fold,92760.0,0.251099,0.355815,0.000000,0.000000,0.000000,0.641127,0.982014
ix_uncertain_early__strict_hybrid_fold,92760.0,0.085066,0.140303,0.000000,0.000000,0.000000,0.157987,0.499961
ix_boundary_mid__strict_hybrid_fold,92760.0,0.098674,0.298225,0.000000,0.000000,0.000000,0.000000,1.000000
ix_margin_x_distance__strict_hybrid_fold,92760.0,211.620967,145.618688,0.000298,97.890343,202.006620,304.834395,1514.266410
ix_entropy_x_progress__strict_hybrid_fold,92760.0,0.573266,0.336786,0.004176,0.284528,0.564621,0.849092,1.504851


## 6. XGBoost Interaction Sweep

In [5]:
"Train compact XGBoost candidates over baseline, raw pseudo-top, and interaction feature sets."

base_params = {
    "tree_method": "hist",
    "random_state": SEED,
    "n_jobs": 4,
    "verbosity": 0,
    "eval_metric": "rmse",
}

candidates = {
    "xgb_pseudohuber_reference": {
        "objective": "reg:pseudohubererror",
        "n_estimators": 650,
        "learning_rate": 0.032,
        "max_depth": 4,
        "min_child_weight": 100,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.15,
        "reg_lambda": 7.0,
        "gamma": 0.05,
    },
    "xgb_square_regularized": {
        "objective": "reg:squarederror",
        "n_estimators": 520,
        "learning_rate": 0.036,
        "max_depth": 5,
        "min_child_weight": 90,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.20,
        "reg_lambda": 8.0,
        "gamma": 0.10,
    },
    "xgb_interaction_shallow": {
        "objective": "reg:pseudohubererror",
        "n_estimators": 760,
        "learning_rate": 0.026,
        "max_depth": 3,
        "min_child_weight": 120,
        "subsample": 0.88,
        "colsample_bytree": 0.88,
        "reg_alpha": 0.25,
        "reg_lambda": 10.0,
        "gamma": 0.10,
    },
}


def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


def feature_sets_for_fold(fold_col):
    pseudo_raw = pseudo_feature_cols_by_fold[fold_col]
    pseudo_uncertainty = pseudo_uncertainty_cols_by_fold[fold_col]
    interactions = interaction_cols_by_fold[fold_col]
    return {
        "safe_base": SAFE_FEATURE_COLS,
        "safe_typewell": [*SAFE_FEATURE_COLS, *TYPEWELL_FEATURE_COLS],
        "pseudo_raw": [*SAFE_FEATURE_COLS, *TYPEWELL_FEATURE_COLS, *pseudo_raw],
        "pseudo_uncertainty": [*SAFE_FEATURE_COLS, *TYPEWELL_FEATURE_COLS, *pseudo_uncertainty],
        "pseudo_interactions": [*SAFE_FEATURE_COLS, *TYPEWELL_FEATURE_COLS, *interactions],
        "pseudo_raw_interactions": [*SAFE_FEATURE_COLS, *TYPEWELL_FEATURE_COLS, *pseudo_raw, *pseudo_uncertainty, *interactions],
    }


def evaluate_xgb_sweep(fold_col):
    fold_rows = []
    pred_parts = []
    importance_parts = []
    y_delta = data["target_delta"].astype(float)
    y_tvt = data["TVT"].astype(float)
    baseline = data["last_known_tvt_feature"].astype(float)
    for feature_set_name, raw_cols in feature_sets_for_fold(fold_col).items():
        feature_cols = list(dict.fromkeys([col for col in raw_cols if col in data.columns]))
        X_all = data[feature_cols].replace([np.inf, -np.inf], np.nan)
        active_candidates = candidates if "interaction" in feature_set_name or feature_set_name in {"safe_typewell", "pseudo_uncertainty"} else {name: candidates[name] for name in ["xgb_pseudohuber_reference", "xgb_square_regularized"]}
        for candidate_name, params in active_candidates.items():
            label = f"{candidate_name}__{feature_set_name}"
            for fold in sorted(data[fold_col].dropna().unique()):
                valid_mask = data[fold_col].eq(fold)
                train_mask = ~valid_mask
                model = XGBRegressor(**base_params, **params)
                model.fit(X_all.loc[train_mask], y_delta.loc[train_mask], verbose=False)
                pred_delta = model.predict(X_all.loc[valid_mask])
                pred_tvt = baseline.loc[valid_mask].to_numpy() + pred_delta
                actual = y_tvt.loc[valid_mask].to_numpy()
                base = baseline.loc[valid_mask].to_numpy()
                fold_rows.append({
                    "run_id": RUN_ID,
                    "fold_column": fold_col,
                    "fold": int(fold),
                    "label": label,
                    "candidate": candidate_name,
                    "feature_set": feature_set_name,
                    "n_train_rows": int(train_mask.sum()),
                    "n_valid_rows": int(valid_mask.sum()),
                    "n_features": int(len(feature_cols)),
                    "rmse": rmse(actual, pred_tvt),
                    "mae": float(mean_absolute_error(actual, pred_tvt)),
                    "r2": float(r2_score(actual, pred_tvt)),
                    "last_known_hold_rmse": rmse(actual, base),
                })
                tmp = data.loc[valid_mask, ["wellname", PRIMARY_FOLD, STRESS_FOLD, "eval_progress", "selected_analog_distance", "combined_stress_score", f"pseudo_top_margin__{fold_col}", f"pseudo_top_entropy__{fold_col}", f"pseudo_mean_std__{fold_col}"]].copy()
                tmp["fold_column"] = fold_col
                tmp["fold"] = int(fold)
                tmp["label"] = label
                tmp["candidate"] = candidate_name
                tmp["feature_set"] = feature_set_name
                tmp["actual_tvt"] = actual
                tmp["pred_tvt"] = pred_tvt
                tmp["baseline_tvt"] = base
                tmp["residual"] = actual - pred_tvt
                pred_parts.append(tmp)
                importance_parts.append(pd.DataFrame({
                    "fold_column": fold_col,
                    "fold": int(fold),
                    "label": label,
                    "candidate": candidate_name,
                    "feature_set": feature_set_name,
                    "feature": feature_cols,
                    "importance": model.feature_importances_,
                }))
    return pd.DataFrame(fold_rows), pd.concat(pred_parts, ignore_index=True), pd.concat(importance_parts, ignore_index=True)


start = time.time()
strict_fold_metrics, strict_predictions, strict_importance = evaluate_xgb_sweep(PRIMARY_FOLD)
stress_fold_metrics, stress_predictions, stress_importance = evaluate_xgb_sweep(STRESS_FOLD)
print(f"elapsed_s={time.time() - start:.1f}")
strict_fold_metrics.head()

elapsed_s=690.8


,run_id,fold_column,fold,label,candidate,feature_set,n_train_rows,n_valid_rows,n_features,rmse,mae,r2,last_known_hold_rmse
0,2026-06-30__pseudo_top_interaction_xgboost,strict_hybrid_fold,1,xgb_pseudohuber_reference__safe_base,xgb_pseudohuber_reference,safe_base,61440,31320,36,14.533310,9.935727,0.999392,16.418628
1,2026-06-30__pseudo_top_interaction_xgboost,strict_hybrid_fold,2,xgb_pseudohuber_reference__safe_base,xgb_pseudohuber_reference,safe_base,61920,30840,36,12.904631,9.268308,0.999588,14.945727
2,2026-06-30__pseudo_top_interaction_xgboost,strict_hybrid_fold,3,xgb_pseudohuber_reference__safe_base,xgb_pseudohuber_reference,safe_base,62160,30600,36,13.003362,9.307869,0.999573,15.349476
3,2026-06-30__pseudo_top_interaction_xgboost,strict_hybrid_fold,1,xgb_square_regularized__safe_base,xgb_square_regularized,safe_base,61440,31320,36,13.791879,9.530099,0.999453,16.418628
4,2026-06-30__pseudo_top_interaction_xgboost,strict_hybrid_fold,2,xgb_square_regularized__safe_base,xgb_square_regularized,safe_base,61920,30840,36,13.263380,9.510448,0.999564,14.945727


## 7. Results And Stress Diagnostics

In [6]:
"Summarize strict-CV and analog-distance stress performance."

strict_summary = (
    strict_fold_metrics.groupby(["label", "candidate", "feature_set"])
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        r2_mean=("r2", "mean"),
        last_known_hold_rmse_mean=("last_known_hold_rmse", "mean"),
        n_valid_rows=("n_valid_rows", "sum"),
        n_features=("n_features", "max"),
    )
    .reset_index()
)
stress_summary = (
    stress_fold_metrics.groupby(["label", "candidate", "feature_set"])
    .agg(
        stress_rmse_mean=("rmse", "mean"),
        stress_rmse_std=("rmse", "std"),
        stress_mae_mean=("mae", "mean"),
        stress_last_known_hold_rmse_mean=("last_known_hold_rmse", "mean"),
        stress_n_valid_rows=("n_valid_rows", "sum"),
    )
    .reset_index()
)
summary = strict_summary.merge(stress_summary, on=["label", "candidate", "feature_set"], how="left")
summary["rmse_delta_vs_hold"] = summary["rmse_mean"] - summary["last_known_hold_rmse_mean"]
summary["rmse_delta_vs_safe_typewell_reference"] = summary["rmse_mean"] - 13.450635286744287
summary["rmse_delta_vs_public_best"] = summary["rmse_mean"] - CONFIG["current_public_best_rmse"]
summary["stress_minus_strict_rmse"] = summary["stress_rmse_mean"] - summary["rmse_mean"]
summary = summary.sort_values("rmse_mean").reset_index(drop=True)
best_row = summary.iloc[0].copy()
best_label = str(best_row["label"])

predictions = pd.concat([strict_predictions, stress_predictions], ignore_index=True)
predictions["progress_zone"] = pd.cut(predictions["eval_progress"], bins=[-0.001, 0.25, 0.50, 0.75, 1.001], labels=["z0", "z1", "z2", "z3"])
predictions["analog_distance_bin"] = pd.qcut(predictions["selected_analog_distance"], q=5, labels=["closest", "near", "middle", "far", "farthest"], duplicates="drop")

zone_summary = (
    predictions.groupby(["fold_column", "label", "progress_zone"], observed=True)
    .apply(lambda group: pd.Series({
        "n_rows": len(group),
        "rmse": rmse(group["actual_tvt"], group["pred_tvt"]),
        "baseline_rmse": rmse(group["actual_tvt"], group["baseline_tvt"]),
        "mean_abs_residual": float(np.abs(group["residual"]).mean()),
        "mean_pseudo_margin": float(group.filter(like="pseudo_top_margin").iloc[:, 0].mean()),
        "mean_pseudo_entropy": float(group.filter(like="pseudo_top_entropy").iloc[:, 0].mean()),
    }), include_groups=False)
    .reset_index()
)
zone_summary["rmse_delta_vs_hold"] = zone_summary["rmse"] - zone_summary["baseline_rmse"]

analog_bin_summary = (
    predictions.groupby(["fold_column", "label", "analog_distance_bin"], observed=True)
    .apply(lambda group: pd.Series({
        "n_rows": len(group),
        "n_wells": group["wellname"].nunique(),
        "rmse": rmse(group["actual_tvt"], group["pred_tvt"]),
        "baseline_rmse": rmse(group["actual_tvt"], group["baseline_tvt"]),
    }), include_groups=False)
    .reset_index()
)
analog_bin_summary["rmse_delta_vs_hold"] = analog_bin_summary["rmse"] - analog_bin_summary["baseline_rmse"]

well_diagnostics = (
    predictions.loc[(predictions["fold_column"].eq(PRIMARY_FOLD)) & (predictions["label"].eq(best_label))]
    .groupby("wellname")
    .agg(
        n_rows=("actual_tvt", "size"),
        strict_fold=(PRIMARY_FOLD, "first"),
        analog_fold=(STRESS_FOLD, "first"),
        mean_selected_analog_distance=("selected_analog_distance", "mean"),
        mean_combined_stress=("combined_stress_score", "mean"),
        mean_residual=("residual", "mean"),
        mean_abs_residual=("residual", lambda series: float(np.abs(series).mean())),
        rmse=("residual", lambda series: float(np.sqrt(np.mean(np.square(series))))),
    )
    .reset_index()
    .sort_values("rmse", ascending=False)
)

importance = pd.concat([strict_importance, stress_importance], ignore_index=True)
importance_summary = (
    importance.groupby(["fold_column", "label", "candidate", "feature_set", "feature"])["importance"].mean().reset_index()
    .sort_values(["fold_column", "label", "importance"], ascending=[True, True, False])
)

print(summary.head(18).to_string(index=False))

                                             label                 candidate             feature_set  rmse_mean  rmse_std  mae_mean  r2_mean  last_known_hold_rmse_mean  n_valid_rows  n_features  stress_rmse_mean  stress_rmse_std  stress_mae_mean  stress_last_known_hold_rmse_mean  stress_n_valid_rows  rmse_delta_vs_hold  rmse_delta_vs_safe_typewell_reference  rmse_delta_vs_public_best  stress_minus_strict_rmse
             xgb_square_regularized__safe_typewell    xgb_square_regularized           safe_typewell  13.474489  0.404583  9.554669 0.999521                  15.571277         92760          41         13.828408         0.538659        10.029363                         15.581239                92760           -2.096788                               0.023854                  -0.997511                  0.353919
          xgb_pseudohuber_reference__safe_typewell xgb_pseudohuber_reference           safe_typewell  13.475668  0.905360  9.511030 0.999518                  15.571277       

## 8. Artifact Export

In [7]:
"Export artifacts, report, and model comparison row."

best_strict_rmse = float(best_row["rmse_mean"])
best_stress_rmse = float(best_row["stress_rmse_mean"])
best_interaction = summary.loc[summary["feature_set"].str.contains("interaction", regex=False)].sort_values("rmse_mean").iloc[0]
best_safe = summary.loc[summary["feature_set"].eq("safe_typewell")].sort_values("rmse_mean").iloc[0]

if best_strict_rmse < 13.450635286744287 - 0.05 and best_stress_rmse <= 13.766527299907088:
    recommendation = "package_pseudo_top_interaction_xgboost"
elif best_strict_rmse < CONFIG["current_public_best_rmse"]:
    recommendation = "iterate_before_package"
else:
    recommendation = "diagnostic_only"

findings = pd.DataFrame([
    {"finding": "best_overall", "detail": best_label, "value": best_strict_rmse, "interpretation": f"Best strict RMSE with analog stress RMSE {best_stress_rmse:.4f}."},
    {"finding": "best_safe_typewell", "detail": str(best_safe["label"]), "value": float(best_safe["rmse_mean"]), "interpretation": "Best same-notebook safe/typewell baseline."},
    {"finding": "best_interaction", "detail": str(best_interaction["label"]), "value": float(best_interaction["rmse_mean"]), "interpretation": "Best candidate using pseudo-top interaction features."},
    {"finding": "decision", "detail": recommendation, "value": best_strict_rmse, "interpretation": "Promotion rule checks strict local reference, stress behavior, and current public context."},
])

paths = {
    "summary": TABLE_DIR / "model__pseudo_top_interaction_xgboost_summary__2026-06-30.csv",
    "fold_metrics": TABLE_DIR / "model__pseudo_top_interaction_xgboost_fold_metrics__2026-06-30.csv",
    "zone_summary": TABLE_DIR / "model__pseudo_top_interaction_xgboost_zone_summary__2026-06-30.csv",
    "analog_bin_summary": TABLE_DIR / "model__pseudo_top_interaction_xgboost_analog_bin_summary__2026-06-30.csv",
    "well_diagnostics": TABLE_DIR / "model__pseudo_top_interaction_xgboost_well_diagnostics__2026-06-30.csv",
    "feature_importance": TABLE_DIR / "model__pseudo_top_interaction_xgboost_feature_importance__2026-06-30.csv",
    "findings": TABLE_DIR / "model__pseudo_top_interaction_xgboost_findings__2026-06-30.csv",
}
exports = {
    "summary": summary,
    "fold_metrics": pd.concat([strict_fold_metrics, stress_fold_metrics], ignore_index=True),
    "zone_summary": zone_summary,
    "analog_bin_summary": analog_bin_summary,
    "well_diagnostics": well_diagnostics,
    "feature_importance": importance_summary,
    "findings": findings,
}
for key, frame in exports.items():
    frame.to_csv(paths[key], index=False)
    shutil.copy2(paths[key], RUN_DIR / paths[key].name)

comparison_path = TABLE_DIR / "model_comparison.csv"
comparison = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()
new_row = pd.DataFrame([{
    "run_id": RUN_ID,
    "model_family": "alignment",
    "model_name": "pseudo_top_interaction_xgboost",
    "best_candidate": best_label,
    "local_cv_rmse": best_strict_rmse,
    "local_cv_mae": float(best_row["mae_mean"]),
    "public_lb_rmse": np.nan,
    "status": recommendation,
    "notes": f"stress_rmse={best_stress_rmse:.4f}; best_safe={float(best_safe['rmse_mean']):.4f}; best_interaction={float(best_interaction['rmse_mean']):.4f}",
    "updated_at": RUN_DATE,
}])
if not comparison.empty and "run_id" in comparison.columns:
    comparison = comparison.loc[comparison["run_id"] != RUN_ID].copy()
comparison = pd.concat([comparison, new_row], ignore_index=True, sort=False)
comparison.to_csv(comparison_path, index=False)
shutil.copy2(comparison_path, RUN_DIR / comparison_path.name)

report_path = REPORT_DIR / "model__pseudo_top_interaction_xgboost__2026-06-30.md"
report = f"""# Pseudo-Top Interaction XGBoost

## Result

Best overall candidate: `{best_label}` with strict-CV RMSE {best_strict_rmse:.4f}, MAE {float(best_row["mae_mean"]):.4f}, and analog-distance stress RMSE {best_stress_rmse:.4f}.

Best same-notebook safe/typewell baseline: `{best_safe["label"]}` with RMSE {float(best_safe["rmse_mean"]):.4f}.

Best pseudo-top interaction candidate: `{best_interaction["label"]}` with RMSE {float(best_interaction["rmse_mean"]):.4f}.

## Interaction Design

Interactions include pseudo-top margin by progress/distance, entropy by analog distance/progress, nearest-top offset by distance, confidence by typewell error, pseudo-top spread by stress, and simple early/late/boundary regime flags.

## Leakage Notes

Direct formation columns are labels only for fold-safe pseudo-top proxy training. Interaction features are derived only from held-out pseudo-top predictions and hidden-test-safe row/typewell/state columns.

## Decision

Recommendation: `{recommendation}`.

## Next Move

If interactions do not improve strict CV, the next route should be Bayesian/mixture priors over stratigraphic regimes rather than more flat feature engineering.
"""
report_path.write_text(report, encoding="utf-8")
shutil.copy2(report_path, RUN_DIR / report_path.name)

print(findings.to_string(index=False))
print(report_path)

           finding                                         detail     value                                                                             interpretation
      best_overall          xgb_square_regularized__safe_typewell 13.474489                                          Best strict RMSE with analog stress RMSE 13.8284.
best_safe_typewell          xgb_square_regularized__safe_typewell 13.474489                                                 Best same-notebook safe/typewell baseline.
  best_interaction xgb_pseudohuber_reference__pseudo_interactions 13.481486                                      Best candidate using pseudo-top interaction features.
          decision                         iterate_before_package 13.474489 Promotion rule checks strict local reference, stress behavior, and current public context.
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\reports\model__pseudo_top_interaction_xgboost__2026-06-30.md


## 9. Findings

In [8]:
"Show headline outputs."

display(summary.head(15))
display(importance_summary.loc[importance_summary["label"].eq(best_label)].head(20))
display(findings)

,label,candidate,feature_set,rmse_mean,rmse_std,mae_mean,r2_mean,last_known_hold_rmse_mean,n_valid_rows,n_features,stress_rmse_mean,stress_rmse_std,stress_mae_mean,stress_last_known_hold_rmse_mean,stress_n_valid_rows,rmse_delta_vs_hold,rmse_delta_vs_safe_typewell_reference,rmse_delta_vs_public_best,stress_minus_strict_rmse
0,xgb_square_regularized__safe_typewell,xgb_square_regularized,safe_typewell,13.474489,0.404583,9.554669,0.999521,15.571277,92760,41,13.828408,0.538659,10.029363,15.581239,92760,-2.096788,0.023854,-0.997511,0.353919
1,xgb_pseudohuber_reference__safe_typewell,xgb_pseudohuber_reference,safe_typewell,13.475668,0.905360,9.511030,0.999518,15.571277,92760,41,13.773051,0.667937,9.832059,15.581239,92760,-2.095608,0.025033,-0.996332,0.297383
2,xgb_pseudohuber_reference__safe_base,xgb_pseudohuber_reference,safe_base,13.480434,0.913153,9.503968,0.999518,15.571277,92760,36,13.850733,0.646157,9.883660,15.581239,92760,-2.090843,0.029799,-0.991566,0.370299
3,xgb_pseudohuber_reference__pseudo_interactions,xgb_pseudohuber_reference,pseudo_interactions,13.481486,0.978312,9.492216,0.999517,15.571277,92760,51,13.868133,0.735750,9.895320,15.581239,92760,-2.089791,0.030851,-0.990514,0.386647
4,xgb_square_regularized__safe_base,xgb_square_regularized,safe_base,13.482319,0.275657,9.531144,0.999522,15.571277,92760,36,13.823441,0.591200,10.016448,15.581239,92760,-2.088957,0.031684,-0.989681,0.341121
5,xgb_pseudohuber_reference__pseudo_uncertainty,xgb_pseudohuber_reference,pseudo_uncertainty,13.505340,0.967509,9.530346,0.999516,15.571277,92760,56,13.923067,0.708820,9.956757,15.581239,92760,-2.065937,0.054705,-0.966660,0.417727
6,xgb_square_regularized__pseudo_interactions,xgb_square_regularized,pseudo_interactions,13.522928,0.513520,9.561848,0.999517,15.571277,92760,51,13.953959,0.601748,10.082099,15.581239,92760,-2.048349,0.072292,-0.949072,0.431031
7,xgb_pseudohuber_reference__pseudo_raw_interact...,xgb_pseudohuber_reference,pseudo_raw_interactions,13.563421,0.997533,9.588551,0.999511,15.571277,92760,81,13.979229,0.691728,9.985338,15.581239,92760,-2.007856,0.112785,-0.908579,0.415808
8,xgb_square_regularized__pseudo_uncertainty,xgb_square_regularized,pseudo_uncertainty,13.565081,0.469766,9.616376,0.999515,15.571277,92760,56,13.913119,0.531943,10.112946,15.581239,92760,-2.006196,0.114446,-0.906919,0.348038
9,xgb_square_regularized__pseudo_raw_interactions,xgb_square_regularized,pseudo_raw_interactions,13.587904,0.581048,9.613213,0.999512,15.571277,92760,81,14.000827,0.415474,10.152826,15.581239,92760,-1.983373,0.137269,-0.884096,0.412923


,fold_column,label,candidate,feature_set,feature,importance
845,analog_distance_fold,xgb_square_regularized__safe_typewell,xgb_square_regularized,safe_typewell,extrapolation_stress_score,0.095194
837,analog_distance_fold,xgb_square_regularized__safe_typewell,xgb_square_regularized,safe_typewell,descriptor_cluster,0.054781
836,analog_distance_fold,xgb_square_regularized__safe_typewell,xgb_square_regularized,safe_typewell,combined_stress_score,0.042955
853,analog_distance_fold,xgb_square_regularized__safe_typewell,xgb_square_regularized,safe_typewell,known_rows,0.039880
835,analog_distance_fold,xgb_square_regularized__safe_typewell,xgb_square_regularized,safe_typewell,analog_stress_score,0.039195
854,analog_distance_fold,xgb_square_regularized__safe_typewell,xgb_square_regularized,safe_typewell,known_tvt_span,0.036764
860,analog_distance_fold,xgb_square_regularized__safe_typewell,xgb_square_regularized,safe_typewell,selected_analog_distance,0.035396
856,analog_distance_fold,xgb_square_regularized__safe_typewell,xgb_square_regularized,safe_typewell,last_known_tvt_feature,0.032957
868,analog_distance_fold,xgb_square_regularized__safe_typewell,xgb_square_regularized,safe_typewell,typewell_mae_last300,0.031771
832,analog_distance_fold,xgb_square_regularized__safe_typewell,xgb_square_regularized,safe_typewell,X,0.031060


,finding,detail,value,interpretation
0,best_overall,xgb_square_regularized__safe_typewell,13.474489,Best strict RMSE with analog stress RMSE 13.8284.
1,best_safe_typewell,xgb_square_regularized__safe_typewell,13.474489,Best same-notebook safe/typewell baseline.
2,best_interaction,xgb_pseudohuber_reference__pseudo_interactions,13.481486,Best candidate using pseudo-top interaction fe...
3,decision,iterate_before_package,13.474489,"Promotion rule checks strict local reference, ..."
